In [ ]:
# separate conda env created for mmpose on linux machine

import cv2
import mmdet
import mmpose
from mmpose.apis.inferencers import MMPoseInferencer

In [ ]:
print(MMPoseInferencer.list_models('mmpose'))

In [ ]:
import re

def parse_sequences(file_path: str) -> dict:
    sequence_dict = {}

    details_pattern = re.compile(
        r"start frame: (\d+), number of frames: (\d+), frames offset: (-?\d+), MoCap data: (\w+)"
    )

    with open(file_path, 'r') as file:
        for line in file:
            if line.strip().startswith('* p'):
                last_key = line.split(' ')[1]
            
            detail_match = details_pattern.search(line.strip())
            if detail_match:
                sequence_dict[last_key] = {
                    "start_frame": int(detail_match.group(1)),
                    "number_of_frames": int(detail_match.group(2)),
                    "frame_offset": int(detail_match.group(3)),
                    "MoCap_data": detail_match.group(4) == "Yes"
                }
                
    return sequence_dict

def get_video_files(sequence_key):
    file_path = './gait3d/ListOfSequences.txt'
    sequence_info = parse_sequences(file_path)[sequence_key]
    avi_file_names =[
        f"c{camera_number}_{(4 - len(str(sequence_info['start_frame']))) * '0' + str(sequence_info['start_frame'])}" 
        for camera_number in range(1, 5)
        ]
    
    avi_seq_paths = [
        f"./gait3d/Sequences/{sequence_key}/Images/{avi_file_name}.avi"
        for avi_file_name in avi_file_names
        ]
    
    return avi_seq_paths
    
file_path = 'gait3d/ListOfSequences.txt'
sequences = parse_sequences(file_path)

In [ ]:
sequences.keys()

In [ ]:
sequences['p1s1']

In [ ]:
video_files = get_video_files('p1s1')
video_files

In [ ]:
inferencer = MMPoseInferencer(pose2d='td-hm_ViTPose-huge_8xb64-210e_coco-256x192', scope='mmpose', device='cuda:0')


cap = cv2.VideoCapture(video_files[0])

while cap.isOpened():
    success, frame = cap.read()

    if not success:
        break

    results_generator = inferencer(frame)
    result = next(results_generator)

    keypoints = result['predictions'][0][0]['keypoints']
    print(keypoints)
    print(len(keypoints))
    print(type(keypoints))
    break

In [ ]:
frame_size = (960, 540)
keypoints_converted = [[dim[0]/frame_size[0], dim[1]/frame_size[1]] for dim in keypoints]
keypoints_converted

In [ ]:
from mmpose.apis.inferencers import MMPoseInferencer
from mmengine.logging import MessageHub
from typing import Literal, Tuple, Sequence
import json
import logging

def prepare_mmpose_dataset(
    inferencer_type: Literal['rtmpose', 'vitpose', 'hrnet'],
    sequences_keys: Sequence[str],
    frame_size: Tuple[int, int] = (960, 540),
):
    logging.getLogger('mmengine').setLevel(logging.ERROR)
    logging.getLogger('mmpose').setLevel(logging.ERROR)

    if inferencer_type == 'rtmpose':
        inferencer = MMPoseInferencer(
            pose2d='rtmpose-l_8xb256-420e_coco-256x192', 
            scope='mmpose', 
            device='cuda:1',
        )
    elif inferencer_type == 'vitpose':
        inferencer = MMPoseInferencer(
            pose2d='td-hm_ViTPose-huge_8xb64-210e_coco-256x192', 
            scope='mmpose', 
            device='cuda:0'
        )
    else:
        inferencer = MMPoseInferencer(
            pose2d='td-hm_hrnet-w48_8xb32-210e_coco-384x288',
            scope='mmpose', 
            device='cuda:1'
        )

    dataset = {}
    
    for sequence_key in sequences_keys:
        sequence_predictions = {}
        print(f"| {sequence_key}", end=" ")
        video_files = get_video_files(sequence_key)
        for file_path in video_files:
            camera = file_path.split('/')[-1].split('_')[0]
            print(camera, end = " ")
            results_generator = inferencer(file_path, show=False, show_progress=False)
            video_predictions = []
            
            for result in results_generator:
                predictions = result['predictions'][0]
                
                if predictions and len(keypoints := predictions[0]['keypoints']) == 17:
                    keypoints_converted = [[dim[0]/frame_size[0], dim[1]/frame_size[1]] for dim in keypoints]
                    video_predictions.append(keypoints_converted)
                else:
                    video_predictions.append([])

            sequence_predictions[camera] = video_predictions
            
        dataset[sequence_key] = sequence_predictions
        with open(f"./datasets/mmpose/dataset_{inferencer_type}.json", 'w') as f:
            json.dump(dataset, f, indent=4)


In [ ]:
import time
start = time.perf_counter()

prepare_mmpose_dataset(inferencer_type = 'rtmpose', sequences_keys = ['p1s1', 'p1s2'])

end = time.perf_counter()
time_rtmpose = (end - start)/2
print(f"Elapsed time: {end - start:.6f} seconds")

In [ ]:
start = time.perf_counter()

prepare_mmpose_dataset(inferencer_type = 'vitpose', sequences_keys = ['p1s1', 'p1s2'])

end = time.perf_counter()
time_vitpose = (end - start)/2
print(f"Elapsed time: {end - start:.6f} seconds")

In [ ]:
start = time.perf_counter()

prepare_mmpose_dataset(inferencer_type = 'vitpose', sequences_keys = ['p1s1', 'p1s2'])

end = time.perf_counter()
time_vitpose = (end - start)/2
print(f"Elapsed time: {end - start:.6f} seconds")

In [ ]:
print(f"Estimated time rtmpose: {166*time_rtmpose/60/60}")
print(f"Estimated time vitpose: {166*time_vitpose/60/60}")

In [ ]:
prepare_mmpose_dataset(inferencer_type = 'rtmpose', sequences_keys = sequences.keys())

In [ ]:
prepare_mmpose_dataset(inferencer_type = 'vitpose', sequences_keys = sequences.keys())

In [ ]:
prepare_mmpose_dataset(inferencer_type = 'hrnet', sequences_keys = sequences.keys())